# 20 Newsgroups Information Retrieval — TF-IDF Retrieval System Demo

**Author:** Reetikesh Choudhury  
**Module:** TF-IDF Retrieval (`src/tfidf.py`)

This notebook demonstrates the **TF-IDF vector retrieval engine** built on the common processed dataset (`data/processed/processed_documents.parquet`).

---

### Pipeline Overview (no data leakage)
1. **Load Processed Corpus** — full parquet (~19,997 documents with `split` column)
2. **Filter to Training Documents** — `df[df['split'] == 'train']` (~15,997 docs, 80%)
3. **Build TF-IDF Index** — `TfidfVectorizer.fit_transform()` on **training docs only**
4. **Query the Index** — cosine similarity against training document matrix
5. **Test Documents** — transformed with the **already-fitted** vectorizer (no refitting)

> ⚠️ **Data Leakage Warning**: Never call `build_tfidf(df)` with the full dataset.
> Always pass `train_df = df[df['split'] == 'train']` to avoid vocabulary leakage from test documents.

In [ ]:
import sys
import time
from pathlib import Path
import pandas as pd

# Add src/ directory to Python path
NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from tfidf import build_tfidf, search_tfidf, get_document, load_processed_dataset

## Step 1: Load Common Processed Dataset

Load the preprocessed Parquet artifact generated by `src/build_dataset.py`.  
This artifact already contains the `split` column (`'train'` / `'test'`) assigned by the stratified 80/20 split.

In [ ]:
t0 = time.perf_counter()
df = load_processed_dataset()
load_time = time.perf_counter() - t0

train_df = df[df['split'] == 'train'].reset_index(drop=True)
test_df  = df[df['split'] == 'test'].reset_index(drop=True)

print(f"Dataset loaded in {load_time:.2f} seconds.")
print(f"Columns: {df.columns.tolist()}")
print()
print(f"Total documents   : {len(df):,}")
print(f"Training documents: {len(train_df):,}  ({100*len(train_df)/len(df):.1f}%)")
print(f"Testing documents : {len(test_df):,}  ({100*len(test_df)/len(df):.1f}%)")

df.head(3)

## Step 2: Build TF-IDF Retrieval Index (Training Documents Only)

Fit the `TfidfVectorizer` on `clean_text` from **training documents only**.  
Test documents are **never** seen during vocabulary/IDF weight computation.

In [ ]:
t0 = time.perf_counter()

# Pass training documents ONLY — vectorizer.fit_transform() runs on these
searcher = build_tfidf(train_df)

build_time = time.perf_counter() - t0

print(f"TF-IDF Index built in {build_time:.2f} seconds.")
print(f"Documents indexed (train only): {searcher.num_documents:,}")
print(f"Vocabulary size               : {searcher.vocab_size:,} terms")
print()
print("NOTE: Vocabulary and IDF weights were computed from training documents only.")
print(f"Test documents ({len(test_df):,}) will be transformed with this fitted vectorizer.")

## Step 3: Transform Test Documents with Fitted Vectorizer

Use `vectorizer.transform()` (not `fit_transform()`) on test documents.  
This applies the training vocabulary — unknown test tokens are silently ignored.

In [ ]:
t0 = time.perf_counter()

test_texts = test_df['clean_text'].fillna('').tolist()
test_matrix = searcher.vectorizer.transform(test_texts)

transform_time = time.perf_counter() - t0

print(f"Test documents transformed in {transform_time:.3f} seconds.")
print(f"Test matrix shape : {test_matrix.shape}")
print(f"  rows = {test_matrix.shape[0]:,} test documents")
print(f"  cols = {test_matrix.shape[1]:,} vocabulary terms (same as training index)")
print()
print("This matrix is used for evaluation — e.g. computing query-document")
print("similarity or evaluating precision/recall over test queries.")

## Step 4: Run Sample Queries

Execute queries across various newsgroup topics and display top-5 retrieved results.  
Retrieval is performed against the **training index**.

In [ ]:
sample_queries = [
    "space shuttle launch moon orbit",
    "3d graphics rendering algorithm ray tracing",
    "public key encryption pgp cryptography security",
    "motorcycle helmet riding safety gears",
    "christian church faith jesus religion",
    "windows driver installation system setup",
    "baseball pitcher home run inning game",
]

for query in sample_queries:
    t_start = time.perf_counter()
    results = searcher.search(query, top_k=5)
    search_ms = (time.perf_counter() - t_start) * 1000

    print(f"\nQuery: '{query}' ({search_ms:.2f} ms)")

    query_display = []
    for res in results:
        doc = get_document(res["doc_id"])
        query_display.append({
            "Rank"    : res["rank"],
            "doc_id"  : res["doc_id"],
            "Score"   : round(res["score"], 4),
            "Split"   : doc["split"] if doc else "N/A",
            "Category": doc["category"] if doc else "N/A",
            "Snippet" : doc["text"].replace("\n", " ").strip()[:60] + "..." if doc else ""
        })

    display_df = pd.DataFrame(query_display)
    display(display_df)

## Step 5: Verify Standardized Output Structure

Verify that `search_tfidf()` returns the exact standardized dictionary format expected by the team:

```json
[
    {
        "doc_id": 123,
        "score": 0.361942,
        "rank": 1
    }
]
```

In [ ]:
# Module-level convenience call — uses the globally cached searcher (train only)
results = search_tfidf("space exploration", top_k=3)

import json
print(json.dumps(results, indent=2))

## Step 6: Split Summary

Quick sanity check confirming the 80/20 split and that the index contains only training documents.

In [ ]:
print('=' * 55)
print('  TF-IDF INDEX SUMMARY')
print('=' * 55)
print(f'  Total documents in parquet  : {len(df):,}')
print(f'  Training docs (index built) : {len(train_df):,}  ({100*len(train_df)/len(df):.1f}%)')
print(f'  Testing  docs (eval only)   : {len(test_df):,}  ({100*len(test_df)/len(df):.1f}%)')
print(f'  Searcher.num_documents      : {searcher.num_documents:,}  (should equal train count)')
print(f'  Vocabulary size             : {searcher.vocab_size:,} terms')
print(f'  Test matrix shape           : {test_matrix.shape}')
print('=' * 55)
assert searcher.num_documents == len(train_df), \
    f'Index contains {searcher.num_documents} docs but train_df has {len(train_df)}'
print('  ✓ Index built on training documents only (no leakage)')